# Inference Configuration
Specify
- path

In [1]:
path = "../../output/protenn2/v5"


In [2]:
import json
import os.path
import pickle

import torch
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader

from src.protenn2.utils import get_train_val_test_paths, get_device

# get file paths

log_path = os.path.join(path, "log")

label_encoder_path = os.path.join(path, "label_encoder.pkl")
model_path = os.path.join(path, "best_model.pt")
with open(os.path.join(path, "params.json"), "r") as f:
    params = json.load(f)
if "input_folder" not in params:
    raise ValueError("input_folder must be specified")
dataset_path = os.path.join("../../", params["input_folder"])
train_path, val_path, test_path = get_train_val_test_paths(dataset_path)


In [3]:
from src.protenn2.utils import calculate_max_protein_length
from src.protenn2.dataset import CathPredPerResidueDataset, create_protein_collate_fn
from src.protenn2.model import CathPredEnn2
from src.protenn2.analysis.cath_hierarchy_mapper import CATHHierarchyMapper

# Initialize objects

device = get_device()
with open(label_encoder_path, "rb") as f:
    label_encoder: LabelEncoder = pickle.load(f)
num_classes = len(label_encoder.classes_)
max_protein_length = calculate_max_protein_length(dataset_path)

model = CathPredEnn2(num_classes=num_classes)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
test_dataset = CathPredPerResidueDataset(test_path, label_encoder=label_encoder,
                                         embedding_dir="../../data/embeddings/protein_embeddings_new")

collate_fn = create_protein_collate_fn(max_protein_length, test_dataset.padding_encoded_id)

test_dataloader = DataLoader(test_dataset, collate_fn=collate_fn)

mapper = CATHHierarchyMapper(label_encoder=label_encoder)

Using MPS (Apple Silicon GPU).
Max protein length: 599
Dataset initialized with 1317 unique proteins.


In [4]:
from src.protenn2.analysis.inference import run_inference

y_true_labels_list, y_pred_confidences_list, protein_chain_id_list = run_inference(model=model,
                                                                                   dataloader=test_dataloader,
                                                                                   padding_encoded_id=test_dataset.padding_encoded_id,
                                                                                   device=device,
                                                                                   return_protein_chain_id=True)

Running inference on 1317 proteins...


Inference Progress: 100%|██████████| 1317/1317 [00:02<00:00, 447.74it/s]

Inference complete. Processed 1317 proteins


# Analysis Configuration

In [5]:
from src.protenn2.utils import call_domains_list

bootstrap_samples = 1000
post_process_kwargs = {"reporting_threshold": 0.1, "region_min_length": 20, "gaussian_sigma": 2}
post_process_func = call_domains_list
# metrics_to_compute = ("accuracy", "f1_score", "jaccard_score", "recall_score", "precision_score",
#                       "segment_overlap_score")

metrics_to_compute = ("segment_overlap_score")

In [6]:
from src.protenn2.analysis.metrics import calculate_metrics_for_cath_levels

all_results = calculate_metrics_for_cath_levels(y_true_labels_list=y_true_labels_list,
                                                y_pred_confidences_list=y_pred_confidences_list, mapper=mapper,
                                                bootstrap_samples=bootstrap_samples,
                                                post_process_func=post_process_func,
                                                post_process_kwargs=post_process_kwargs,
                                                metrics_to_compute=metrics_to_compute)

---- Computing Metrics for hierarchy: C
----- Segment Overlap Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:56<00:00, 17.58it/s]


{'mean': np.float64(0.5355620101722546), 'ci_lower': np.float64(0.5135025737048332), 'ci_upper': np.float64(0.5566080116349764), 'alpha': 0.05}
----- Segment Overlap Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:49<00:00, 20.31it/s]


{'mean': np.float64(0.6116769153715313), 'ci_lower': np.float64(0.5869680366559386), 'ci_upper': np.float64(0.636796982535919), 'alpha': 0.05}
---- Computing Metrics for hierarchy: A
----- Segment Overlap Score Metrics -----


...

{'mean': np.float64(0.5888565159786577), 'ci_lower': np.float64(0.5630362474316571), 'ci_upper': np.float64(0.6138654159788861), 'alpha': 0.05}
---- Computing Metrics for hierarchy: H
----- Segment Overlap Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [01:02<00:00, 16.08it/s]


{'mean': np.float64(0.43263473104817274), 'ci_lower': np.float64(0.41114079954703053), 'ci_upper': np.float64(0.45357446168575627), 'alpha': 0.05}
----- Segment Overlap Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:54<00:00, 18.39it/s]

{'mean': np.float64(0.5758845668177291), 'ci_lower': np.float64(0.5504957903360751), 'ci_upper': np.float64(0.6017545348042315), 'alpha': 0.05}


# All results

In [7]:
all_results

{'raw_C': {'segment_overlap_score': {'mean': np.float64(0.5355620101722546),
   'ci_lower': np.float64(0.5135025737048332),
   'ci_upper': np.float64(0.5566080116349764),
   'alpha': 0.05}},
 'post_C': {'segment_overlap_score': {'mean': np.float64(0.6116769153715313),
   'ci_lower': np.float64(0.5869680366559386),
   'ci_upper': np.float64(0.636796982535919),
   'alpha': 0.05}},
 'raw_A': {'segment_overlap_score': {'mean': np.float64(0.4756124534293524),
   'ci_lower': np.float64(0.45398430204341134),
   'ci_upper': np.float64(0.4965644792638824),
   'alpha': 0.05}},
 'post_A': {'segment_overlap_score': {'mean': np.float64(0.5997696546973686),
   'ci_lower': np.float64(0.5747136256214578),
   'ci_upper': np.float64(0.6250051714885909),
   'alpha': 0.05}},
 'raw_T': {'segment_overlap_score': {'mean': np.float64(0.4524655826109164),
   'ci_lower': np.float64(0.43018261495059346),
   'ci_upper': np.float64(0.4738909582176198),
   'alpha': 0.05}},
 'post_T': {'segment_overlap_score': {'mea

In [8]:
with open(os.path.join(path, "metrics_test_new.json"), "w") as f:
    json.dump(all_results, f)